# AIO Presence & Overlap Analysis
Loads all `serp_raw_*.json` files and analyses when Google AI Overviews appear and how much they overlap with organic results.

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from common import (
    load_data,
    set_plot_style,
    PALETTE,
    ROOT,
)

sns.set_theme(style="whitegrid")
set_plot_style()
df = load_data()

## 0. Query counts

In [ ]:
n_records = len(df)
n_unique_queries = df["query"].nunique()

print(f"Total records (SERP requests) : {n_records}")
print(f"Unique queries                : {n_unique_queries}")

print("\nBy stance:")
print(df["stance"].value_counts().reindex(["Pro", "Neutral", "Con"]).to_string())

print("\nBy leaning:")
print(df["pro_leaning"].value_counts().reindex(["Left", "Right"]).to_string())

print("\nBy leaning × stance:")
print(
    df.groupby(["pro_leaning", "stance"])
    .size()
    .reindex(pd.MultiIndex.from_product([["Left", "Right"], ["Pro", "Neutral", "Con"]]))
    .to_string()
)

## 1. AIO presence per topic

In [ ]:
topic_stats = (
    df.groupby("topic")["has_ai_overview"]
    .agg(total="count", aio_present="sum")
    .assign(
        aio_absent=lambda x: x["total"] - x["aio_present"],
        aio_rate=lambda x: x["aio_present"] / x["total"],
    )
    .sort_values("aio_rate", ascending=False)
)
display(topic_stats.style.format({"aio_rate": "{:.0%}"}))

fig, ax = plt.subplots(figsize=(10, 4))
x = range(len(topic_stats))
ax.bar(x, topic_stats["aio_present"], label="AIO present", color="steelblue")
ax.bar(
    x,
    topic_stats["aio_absent"],
    bottom=topic_stats["aio_present"],
    label="AIO absent",
    color="#d9534f",
)
ax.set_xticks(list(x))
ax.set_xticklabels(topic_stats.index, rotation=30, ha="right")
ax.set_ylabel("Number of requests")
ax.set_title("AIO presence per topic")
ax.legend()
plt.tight_layout()
plt.show()

## 1b. AIO fetch attempts — first try, retry, or never

`aio_fetch_attempts` records how many SerpAPI requests were made before AIO appeared (or the collector gave up). The collector retries up to 2 times ([serpapi_collector.py](../scripts/serpapi_collector.py)), so a query either got AIO on the **first** try, got it on a **retry**, or **never** got it after the max attempts.

In [ ]:
def _attempt_bucket(row):
    if row["aio_fetch_attempts"] <= 1:
        return "First attempt"
    elif row["has_ai_overview"]:
        return "Retry succeeded"
    else:
        return "Never (gave up)"


df["aio_attempt_bucket"] = df.apply(_attempt_bucket, axis=1)
ATTEMPT_ORDER = ["First attempt", "Retry succeeded", "Never (gave up)"]

attempt_counts = (
    df["aio_attempt_bucket"].value_counts().reindex(ATTEMPT_ORDER).fillna(0).astype(int)
)
print(attempt_counts.to_string())
print(f"\nShare of all queries: \n{(attempt_counts / len(df)).round(3).to_string()}")

fig, ax = plt.subplots(figsize=(6, 4))
colors = ["#2ecc71", "#f39c12", "#e74c3c"]
bars = ax.bar(ATTEMPT_ORDER, attempt_counts.values, color=colors)
ax.bar_label(bars, padding=3)
ax.set_ylabel("Number of queries")
ax.set_title("AIO fetch outcome — first attempt vs retry vs never")
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.show()

## 2. AIO presence per stance (Pro / Neutral / Con)

In [ ]:
df_stance = df[df["stance"].notna()]

if df_stance.empty:
    print("No stance data yet — run generate_queries.py + serpapi_collector.py first.")
else:
    stance_stats = (
        df_stance.groupby("stance")["has_ai_overview"]
        .agg(total="count", aio_present="sum")
        .assign(aio_rate=lambda x: x["aio_present"] / x["total"])
        .reindex(["Pro", "Neutral", "Con"])
    )
    display(stance_stats.style.format({"aio_rate": "{:.0%}"}))

    fig, ax = plt.subplots(figsize=(6, 4))
    for i, (stance, row) in enumerate(stance_stats.iterrows()):
        ax.bar(i, row["aio_rate"], color=PALETTE.get(stance, "grey"), label=stance)
    ax.set_xticks(range(len(stance_stats)))
    ax.set_xticklabels(stance_stats.index)
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
    ax.set_ylabel("AIO presence rate")
    ax.set_title("AIO rate by query stance")
    ax.set_ylim(0, 1)
    plt.tight_layout()
    plt.show()

## 2b. AIO presence by political leaning × stance

Breaks down AIO appearance rate across the six combinations of `pro_leaning` (Left / Right) and `stance` (Pro / Neutral / Con).

In [ ]:
df_lean = df[df["pro_leaning"].notna() & df["stance"].notna()].copy()

if df_lean.empty:
    print("No pro_leaning / stance data yet.")
else:
    LEANING_ORDER = ["Left", "Right"]
    STANCE_ORDER = ["Pro", "Neutral", "Con"]
    LEANING_COLORS = {"Left": "#c0392b", "Right": "#2471a3"}
    STANCE_ALPHA = {"Pro": 1.0, "Neutral": 0.55, "Con": 0.3}

    pivot = (
        df_lean.groupby(["pro_leaning", "stance"])["has_ai_overview"]
        .agg(total="count", aio_present="sum")
        .assign(aio_rate=lambda x: x["aio_present"] / x["total"])
        .reindex(
            pd.MultiIndex.from_product(
                [LEANING_ORDER, STANCE_ORDER], names=["pro_leaning", "stance"]
            )
        )
    )
    display(pivot.style.format({"aio_rate": "{:.0%}"}))

    # ── Heatmap ────────────────────────────────────────────────────────
    heat = (
        pivot["aio_rate"]
        .unstack("stance")
        .reindex(index=LEANING_ORDER, columns=STANCE_ORDER)
    )
    fig2, ax2 = plt.subplots(figsize=(5, 2.5))
    import seaborn as sns

    sns.heatmap(
        heat.astype(float),
        annot=True,
        fmt=".0%",
        cmap="RdBu_r",
        vmin=0,
        vmax=1,
        linewidths=0.5,
        ax=ax2,
        cbar_kws={"label": "AIO rate"},
    )
    ax2.set_xlabel("Stance")
    ax2.set_ylabel("Political leaning")
    ax2.set_title("AIO presence rate — leaning × stance")
    plt.tight_layout()
    plt.show()

    # ── Queries where AIO did not activate ────────────────────────────
    no_aio = df_lean[~df_lean["has_ai_overview"]][
        ["query", "topic", "subtopic", "pro_leaning", "stance"]
    ].sort_values(["pro_leaning", "stance", "topic"])
    print(
        f"\nQueries with no AIO: {len(no_aio)} / {len(df_lean)} ({len(no_aio) / len(df_lean):.1%})"
    )
    display(no_aio.reset_index(drop=True))

In [ ]:
# ── 2c. AIO presence: neutral vs polarized (Pro/Con averaged) × leaning ────
if df_lean.empty:
    print("No pro_leaning / stance data yet.")
else:
    rate_by_stance = (
        df_lean.groupby(["pro_leaning", "stance"])["has_ai_overview"]
        .mean()
        .rename("aio_rate")
        .reset_index()
    )

    neutral = (
        rate_by_stance[rate_by_stance["stance"] == "Neutral"]
        .set_index("pro_leaning")["aio_rate"]
        .rename("neutral")
    )
    polarized = (
        rate_by_stance[rate_by_stance["stance"].isin(["Pro", "Con"])]
        .groupby("pro_leaning")["aio_rate"]
        .mean()
        .rename("polarized")
    )

    heat_np = pd.concat([neutral, polarized], axis=1).reindex(LEANING_ORDER)

    fig3, ax3 = plt.subplots(figsize=(5, 2.5))
    sns.heatmap(
        heat_np.astype(float),
        annot=True,
        fmt=".0%",
        cmap="RdBu_r",
        vmin=0,
        vmax=1,
        linewidths=0.5,
        ax=ax3,
        cbar_kws={"label": "AIO rate"},
    )
    ax3.set_xlabel("Query type")
    ax3.set_ylabel("Political leaning")
    ax3.set_title("AIO presence rate — leaning × neutral vs polarized (Pro/Con avg)")
    plt.tight_layout()
    plt.show()

## 3. Consistency — repeated queries
For every query run more than once, did AIO appear consistently?

In [ ]:
consistency = (
    df.groupby("query")["has_ai_overview"]
    .agg(runs="count", aio_count="sum")
    .query("runs > 1")
    .assign(
        aio_rate=lambda x: x["aio_count"] / x["runs"],
        consistent=lambda x: (x["aio_count"] == x["runs"]) | (x["aio_count"] == 0),
        verdict=lambda x: x.apply(
            lambda r: (
                "always"
                if r["aio_count"] == r["runs"]
                else ("never" if r["aio_count"] == 0 else "inconsistent")
            ),
            axis=1,
        ),
    )
    .sort_values(["verdict", "query"])
)

print(consistency["verdict"].value_counts().to_string())
display(consistency)

In [ ]:
inconsistent = consistency[consistency["verdict"] == "inconsistent"]
if inconsistent.empty:
    print("No inconsistent queries.")
else:
    print(f"{len(inconsistent)} queries with inconsistent AIO:")
    display(inconsistent[["runs", "aio_count", "aio_rate"]])
    detail = df[df["query"].isin(inconsistent.index)][
        ["query", "timestamp_utc", "has_ai_overview"]
    ].sort_values(["query", "timestamp_utc"])
    display(detail)

## 4. AIO presence per subtopic

In [ ]:
df_sub = df[df["subtopic"].notna() & df["pro_leaning"].notna() & df["stance"].notna()]

if df_sub.empty:
    print("No subtopic data yet.")
else:
    import numpy as np

    LEANING_ORDER = ["Left", "Right"]
    LEANING_COLORS = {"Left": "#c0392b", "Right": "#2471a3"}
    STANCE_ORDER = ["Pro", "Neutral", "Con"]
    STANCE_ALPHA = {"Pro": 1.0, "Neutral": 0.55, "Con": 0.3}
    STANCE_LABELS = {"Pro": "Pros", "Neutral": "Neutral", "Con": "Cons"}
    COMBOS = [(lean, stance) for lean in LEANING_ORDER for stance in STANCE_ORDER]

    overall_order = (
        df_sub.groupby(["topic", "subtopic"])["has_ai_overview"]
        .mean()
        .reset_index()
        .sort_values(["topic", "has_ai_overview"], ascending=[True, False])
    )

    subtopic_stats = (
        df_sub.groupby(["topic", "subtopic", "pro_leaning", "stance"])[
            "has_ai_overview"
        ]
        .agg(total="count", aio_present="sum")
        .assign(aio_rate=lambda x: x["aio_present"] / x["total"])
        .reset_index()
    )

    n_combos = len(COMBOS)
    bar_height = 0.8 / n_combos
    combo_offsets = (
        np.linspace(-(n_combos - 1) / 2, (n_combos - 1) / 2, n_combos) * bar_height
    )

    for topic in overall_order["topic"].unique():
        subtopics = overall_order.loc[
            overall_order["topic"] == topic, "subtopic"
        ].tolist()
        group = subtopic_stats[subtopic_stats["topic"] == topic]

        y = np.arange(len(subtopics))

        fig, ax = plt.subplots(figsize=(10, max(3, len(subtopics) * 0.9)))
        for (lean, stance), offset in zip(COMBOS, combo_offsets):
            combo_group = group[
                (group["pro_leaning"] == lean) & (group["stance"] == stance)
            ].set_index("subtopic")
            rates = [
                combo_group.loc[s, "aio_rate"] if s in combo_group.index else 0
                for s in subtopics
            ]
            ax.barh(
                y + offset,
                rates,
                bar_height,
                color=LEANING_COLORS[lean],
                alpha=STANCE_ALPHA[stance],
                label=f"{lean} · {STANCE_LABELS[stance]}",
                edgecolor="white",
                linewidth=0.3,
            )

        ax.set_yticks(y)
        ax.set_yticklabels(subtopics)
        ax.set_xlim(0, 1.15)
        ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
        ax.set_title(f"AIO rate by subtopic × leaning × stance — {topic}")
        ax.invert_yaxis()
        ax.legend(
            title="Leaning · Stance",
            bbox_to_anchor=(1.01, 1),
            loc="upper left",
            fontsize=8,
        )
        plt.tight_layout()
        plt.show()

## 5. AIO–Organic Overlap
`aio_organic_overlap` = share of AIO-cited domains that also appear in the organic top-10.  
Only computed for records where AIO was present.

In [ ]:
df_aio = df[df["has_ai_overview"] & df["aio_organic_overlap"].notna()].copy()
print(f"Records with AIO + overlap score: {len(df_aio)}")
print(df_aio["aio_organic_overlap"].describe().to_string())

In [ ]:
# ── 5a. Per topic ─────────────────────────────────────────────────────────────
topics = sorted(df_aio["topic"].dropna().unique())

fig, ax = plt.subplots(figsize=(max(6, len(topics) * 1.5), 5))
sns.boxplot(
    data=df_aio,
    x="topic",
    y="aio_organic_overlap",
    order=topics,
    color="steelblue",
    width=0.5,
    ax=ax,
)
sns.stripplot(
    data=df_aio,
    x="topic",
    y="aio_organic_overlap",
    order=topics,
    color="black",
    size=6,
    alpha=0.5,
    jitter=True,
    ax=ax,
)
ax.set_xticklabels(topics, rotation=30, ha="right", fontsize=16)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
ax.set_ylabel("AIO–organic overlap")
ax.set_xlabel("")
ax.set_title("AIO–organic overlap per topic", fontsize=20)
ax.set_ylim(-0.05, 1.05)
plt.tight_layout()
plt.show()

In [ ]:
# ── 5b. Per subtopic (one plot per topic) ─────────────────────────────────────
df_aio_sub = df_aio[df_aio["subtopic"].notna()]

if df_aio_sub.empty:
    print("No subtopic data with overlap scores yet.")
else:
    for topic, group in df_aio_sub.groupby("topic"):
        subtopics = (
            group.groupby("subtopic")["aio_organic_overlap"]
            .median()
            .sort_values(ascending=False)
            .index.tolist()
        )
        fig, ax = plt.subplots(figsize=(11, max(4, len(subtopics) * 0.7)))
        sns.boxplot(
            data=group,
            y="subtopic",
            x="aio_organic_overlap",
            order=subtopics,
            color="steelblue",
            width=0.5,
            orient="h",
            ax=ax,
        )
        sns.stripplot(
            data=group,
            y="subtopic",
            x="aio_organic_overlap",
            order=subtopics,
            color="black",
            size=4,
            alpha=0.5,
            jitter=True,
            orient="h",
            ax=ax,
        )
        ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
        ax.set_xlim(-0.05, 1.05)
        ax.set_xlabel("AIO–organic overlap")
        ax.set_ylabel("")
        ax.set_title(f"AIO–organic overlap per subtopic — {topic}")
        plt.tight_layout()
        plt.show()

In [ ]:
# ── 5c. Per stance ────────────────────────────────────────────────────────────
df_aio_stance = df_aio[df_aio["stance"].notna()]

if df_aio_stance.empty:
    print("No stance data with overlap scores yet.")
else:
    order = ["Pro", "Neutral", "Con"]
    fig, ax = plt.subplots(figsize=(7, 5))
    sns.boxplot(
        data=df_aio_stance,
        x="stance",
        y="aio_organic_overlap",
        order=order,
        palette=PALETTE,
        width=0.5,
        ax=ax,
    )
    sns.stripplot(
        data=df_aio_stance,
        x="stance",
        y="aio_organic_overlap",
        order=order,
        color="black",
        size=4,
        alpha=0.5,
        jitter=True,
        ax=ax,
    )
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
    ax.set_ylabel("AIO–organic overlap")
    ax.set_xlabel("Query stance")
    ax.set_title("AIO–organic overlap by stance")
    ax.set_ylim(-0.05, 1.05)
    plt.tight_layout()
    plt.show()

    print("\nMedian overlap per stance:")
    print(df_aio_stance.groupby("stance")["aio_organic_overlap"].describe().round(2))

In [ ]:
# ── 5d. Combined: subtopic × stance (one facet grid per topic) ────────────────
df_aio_full = df_aio[df_aio["subtopic"].notna() & df_aio["stance"].notna()]

if df_aio_full.empty:
    print("No combined subtopic+stance data yet.")
else:
    for topic, group in df_aio_full.groupby("topic"):
        subtopics = (
            group.groupby("subtopic")["aio_organic_overlap"]
            .median()
            .sort_values(ascending=False)
            .index.tolist()
        )
        fig, ax = plt.subplots(figsize=(12, max(4, len(subtopics) * 0.8)))
        sns.boxplot(
            data=group,
            y="subtopic",
            x="aio_organic_overlap",
            hue="stance",
            order=subtopics,
            hue_order=["Pro", "Neutral", "Con"],
            palette=PALETTE,
            width=0.6,
            orient="h",
            ax=ax,
        )
        ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
        ax.set_xlim(-0.05, 1.05)
        ax.set_xlabel("AIO–organic overlap")
        ax.set_ylabel("")
        ax.set_title(f"AIO–organic overlap by subtopic × stance — {topic}")
        legend = ax.legend(title="stance", bbox_to_anchor=(1.01, 1), loc="upper left")
        for t in legend.get_texts():
            t.set_text(STANCE_LABELS.get(t.get_text(), t.get_text()))
        plt.tight_layout()
        plt.show()

In [ ]:
# ── 5e. Combined: left subtopics vs right subtopics × stance (one facet grid per topic) ──
df_aio_lean_full = df_aio[
    df_aio["subtopic"].notna()
    & df_aio["stance"].notna()
    & df_aio["pro_leaning"].notna()
]

if df_aio_lean_full.empty:
    print("No combined subtopic+stance+leaning data yet.")
else:
    LEANING_ORDER = ["Left", "Right"]
    STANCE_ORDER = ["Pro", "Neutral", "Con"]
    STANCE_LABELS = {"Pro": "Pros", "Neutral": "Neutral", "Con": "Cons"}

    for topic, group in df_aio_lean_full.groupby("topic"):
        subtopics = (
            group.groupby("subtopic")["aio_organic_overlap"]
            .median()
            .sort_values(ascending=False)
            .index.tolist()
        )
        fig, axes = plt.subplots(
            1, 2, figsize=(16, max(4, len(subtopics) * 0.8)), sharey=True
        )
        for ax, lean in zip(axes, LEANING_ORDER):
            lean_group = group[group["pro_leaning"] == lean]
            sns.boxplot(
                data=lean_group,
                y="subtopic",
                x="aio_organic_overlap",
                hue="stance",
                order=subtopics,
                hue_order=STANCE_ORDER,
                palette=PALETTE,
                width=0.6,
                orient="h",
                ax=ax,
            )
            ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
            ax.set_xlim(-0.05, 1.05)
            ax.set_xlabel("AIO–organic overlap")
            ax.set_ylabel("")
            ax.set_title(lean.capitalize())
            if ax.get_legend() is not None and ax is not axes[0]:
                ax.get_legend().remove()
        legend = axes[0].legend(title="stance", loc="lower right")
        for t in legend.get_texts():
            t.set_text(STANCE_LABELS.get(t.get_text(), t.get_text()))
        fig.suptitle(
            f"AIO–organic overlap by left subtopics-right subtopics × stance — {topic}"
        )
        plt.tight_layout()
        plt.show()

In [ ]:
# ── 5f. Overall: stance × leaning (aggregated across all topics) ──────────────
df_aio_stance_lean = df_aio[df_aio["stance"].notna() & df_aio["pro_leaning"].notna()]

if df_aio_stance_lean.empty:
    print("No stance+leaning data with overlap scores yet.")
else:
    STANCE_ORDER = ["Pro", "Neutral", "Con"]
    LEANING_ORDER = ["Left", "Right"]
    LEANING_COLORS = {"Left": "#c0392b", "Right": "#2471a3"}

    fig, ax = plt.subplots(figsize=(8, 5))
    sns.boxplot(
        data=df_aio_stance_lean,
        x="stance",
        y="aio_organic_overlap",
        hue="pro_leaning",
        order=STANCE_ORDER,
        hue_order=LEANING_ORDER,
        palette=LEANING_COLORS,
        width=0.6,
        ax=ax,
    )
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
    ax.set_ylabel("AIO–organic overlap")
    ax.set_xlabel("Query stance")
    ax.set_title("AIO–organic overlap by stance × political leaning (all topics)")
    ax.set_ylim(-0.05, 1.05)
    ax.legend(title="Leaning")
    plt.tight_layout()
    plt.show()

    print("\nMedian overlap per stance × leaning:")
    print(
        df_aio_stance_lean.groupby(["stance", "pro_leaning"])["aio_organic_overlap"]
        .describe()
        .round(2)
    )

In [ ]:
# ── 5g. Per topic: subtopics merged by leaning × stance ───────────────────────
df_aio_topic_lean_stance = df_aio[
    df_aio["stance"].notna() & df_aio["pro_leaning"].notna()
]

if df_aio_topic_lean_stance.empty:
    print("No stance+leaning data with overlap scores yet.")
else:
    LEANING_ORDER = ["Left", "Right"]
    STANCE_ORDER = ["Pro", "Neutral", "Con"]
    STANCE_LABELS = {"Pro": "Pros", "Neutral": "Neutral", "Con": "Cons"}

    for topic, group in df_aio_topic_lean_stance.groupby("topic"):
        fig, ax = plt.subplots(figsize=(7, 5))
        sns.boxplot(
            data=group,
            x="pro_leaning",
            y="aio_organic_overlap",
            hue="stance",
            order=LEANING_ORDER,
            hue_order=STANCE_ORDER,
            palette=PALETTE,
            width=0.6,
            ax=ax,
        )
        ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
        ax.set_ylim(-0.05, 1.05)
        ax.set_xlabel("Political leaning (subtopics merged)")
        ax.set_ylabel("AIO–organic overlap")
        ax.set_title(
            f"AIO–organic overlap by leaning × stance (subtopics merged) — {topic}"
        )
        legend = ax.legend(title="stance")
        for t in legend.get_texts():
            t.set_text(STANCE_LABELS.get(t.get_text(), t.get_text()))
        plt.tight_layout()
        plt.show()

In [ ]:
# ── 5h. Source overlap per subtopic across Pro/Con/Neutral formulations ──
import numpy as np
from itertools import combinations


def _domain_set(domains_json):
    """Parse a JSON-encoded domain list into a set (empty set on missing/invalid)."""
    if not domains_json or isinstance(domains_json, float):
        return set()
    try:
        domains = (
            json.loads(domains_json) if isinstance(domains_json, str) else domains_json
        )
    except Exception:
        return set()
    return set(d for d in domains if d)


def _jaccard(a, b):
    union = a | b
    return len(a & b) / len(union) if union else None


def _has_real_aio_content(domain_set, aio_text):
    """True if the AIO actually returned something usable (sources or text).

    SerpApi sometimes returns an AIO "shell" (has_ai_overview=True) whose
    second-stage content fetch silently failed — no text_blocks, no
    references. Those rows must not be treated as a real, zero-source AIO:
    doing so makes every other stance look like it has 0% source overlap
    with them, which is a collection failure, not a finding.
    """
    return bool(domain_set) or (isinstance(aio_text, str) and aio_text.strip() != "")


STANCE_ORDER = ["Pro", "Neutral", "Con"]
STANCE_PAIRS = list(combinations(STANCE_ORDER, 2))
PAIR_COLORS = {
    "Pro vs Neutral": "#8e44ad",
    "Pro vs Con": "#f39c12",
    "Neutral vs Con": "#16a085",
}
PAIR_DISPLAY = {
    "Pro vs Neutral": "Pros vs Neutral",
    "Pro vs Con": "Pros vs Cons",
    "Neutral vs Con": "Neutral vs Cons",
}

df_aio_src = df[
    df["has_ai_overview"] & df["subtopic"].notna() & df["stance"].notna()
].copy()
df_aio_src["domain_set"] = df_aio_src["aio_domains"].apply(_domain_set)
df_aio_src["has_real_content"] = df_aio_src.apply(
    lambda r: _has_real_aio_content(r["domain_set"], r.get("aio_text")), axis=1
)

n_stub = (~df_aio_src["has_real_content"]).sum()
if n_stub:
    print(
        f"Note: {n_stub} AIO rows have has_ai_overview=True but no text/sources "
        "(failed second-stage fetch) — excluded from source-overlap comparisons below."
    )

if df_aio_src.empty:
    print("No AIO source data with subtopic+stance yet.")
else:
    rows = []
    for (topic, subtopic), group in df_aio_src.groupby(["topic", "subtopic"]):
        stance_domains = {}
        stance_has_data = {}
        for s in STANCE_ORDER:
            real = group.loc[
                (group["stance"] == s) & group["has_real_content"], "domain_set"
            ]
            stance_has_data[s] = not real.empty
            stance_domains[s] = set().union(*real.tolist()) if not real.empty else set()
        for s1, s2 in STANCE_PAIRS:
            overlap = (
                _jaccard(stance_domains[s1], stance_domains[s2])
                if stance_has_data[s1] and stance_has_data[s2]
                else None
            )
            rows.append(
                {
                    "topic": topic,
                    "subtopic": subtopic,
                    "pair": f"{s1} vs {s2}",
                    "jaccard": overlap,
                }
            )

    overlap_df = pd.DataFrame(rows).dropna(subset=["jaccard"])

    if overlap_df.empty:
        print("No overlapping stance formulations to compare yet.")
    else:
        pair_labels = [f"{s1} vs {s2}" for s1, s2 in STANCE_PAIRS]
        n_pairs = len(pair_labels)
        bar_h = 0.8 / n_pairs
        offsets = np.linspace(-(n_pairs - 1) / 2, (n_pairs - 1) / 2, n_pairs) * bar_h

        for topic, tgroup in overlap_df.groupby("topic"):
            subtopics = (
                tgroup.groupby("subtopic")["jaccard"]
                .mean()
                .sort_values(ascending=False)
                .index.tolist()
            )
            pivot = tgroup.pivot(index="subtopic", columns="pair", values="jaccard")

            y = np.arange(len(subtopics))
            fig, ax = plt.subplots(figsize=(9, max(3, len(subtopics) * 0.6)))
            for pair, offset in zip(pair_labels, offsets):
                vals = (
                    pivot[pair].reindex(subtopics).fillna(0).values
                    if pair in pivot.columns
                    else np.zeros(len(subtopics))
                )
                ax.barh(
                    y + offset,
                    vals,
                    bar_h,
                    color=PAIR_COLORS[pair],
                    label=PAIR_DISPLAY[pair],
                    edgecolor="white",
                    linewidth=0.3,
                )
            ax.set_yticks(y)
            ax.set_yticklabels(subtopics)
            ax.invert_yaxis()
            ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
            ax.set_xlim(0, 1.05)
            ax.set_xlabel("Jaccard overlap of AIO-cited domains")
            ax.set_title(f"AIO source overlap between stance formulations — {topic}")
            ax.legend(
                title="Formulation pair",
                bbox_to_anchor=(1.01, 1),
                loc="upper left",
                fontsize=8,
            )
            plt.tight_layout()
            plt.show()

In [ ]:
# ── 5i. Topic-level average source overlap between formulations ──────────────
import numpy as np
from itertools import combinations

if overlap_df.empty:
    print("No overlap data to aggregate yet.")
else:
    STANCE_ORDER = ["Pro", "Neutral", "Con"]
    pair_labels = [f"{s1} vs {s2}" for s1, s2 in combinations(STANCE_ORDER, 2)]
    PAIR_COLORS = {
        "Pro vs Neutral": "#8e44ad",
        "Pro vs Con": "#f39c12",
        "Neutral vs Con": "#16a085",
    }
    PAIR_DISPLAY = {
        "Pro vs Neutral": "Pros vs Neutral",
        "Pro vs Con": "Pros vs Cons",
        "Neutral vs Con": "Neutral vs Cons",
    }

    topics_order = sorted(overlap_df["topic"].unique())

    topic_pivot = (
        overlap_df.groupby(["topic", "pair"])["jaccard"]
        .mean()
        .reset_index()
        .pivot(index="topic", columns="pair", values="jaccard")
        .reindex(topics_order)
    )

    n_pairs = len(pair_labels)
    bar_w = 0.8 / n_pairs
    offsets = np.linspace(-(n_pairs - 1) / 2, (n_pairs - 1) / 2, n_pairs) * bar_w
    x = np.arange(len(topics_order))

    fig, ax = plt.subplots(figsize=(max(10, len(topics_order) * 1.4), 5))
    for pair, offset in zip(pair_labels, offsets):
        vals = (
            topic_pivot[pair].values
            if pair in topic_pivot.columns
            else np.zeros(len(topics_order))
        )
        ax.bar(
            x + offset,
            vals,
            bar_w,
            color=PAIR_COLORS[pair],
            label=PAIR_DISPLAY[pair],
            edgecolor="white",
            linewidth=0.3,
        )
    ax.set_xticks(x)
    ax.set_xticklabels(topics_order, rotation=30, ha="right")
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
    ax.set_ylim(0, 1.05)
    ax.set_ylabel("Avg. Jaccard overlap of AIO-cited domains")
    ax.set_title(
        "AIO source overlap between stance formulations — per topic (subtopics averaged)"
    )
    ax.legend(title="Formulation pair", fontsize=8)
    plt.tight_layout()
    plt.show()

In [ ]:
# ── 5j. Baseline: source similarity within the same subtopic × stance ────────
# How similar are AIO-cited domains across different query phrasings that share
# the same subtopic and the same stance? This is the "noise floor" against which
# the between-stance overlap (formulation pairs, next section) should be read —
# if pro/con queries for a subtopic already cite different sources amongst
# themselves, low overlap between stances is less informative.
import numpy as np
from itertools import combinations
from pathlib import Path

STANCE_ORDER = ["Pro", "Neutral", "Con"]
LEANING_ORDER = ["Left", "Right"]
STANCE_LABELS = {"Pro": "Pros", "Neutral": "Neutral", "Con": "Cons"}

baseline_rows = []
for (topic, subtopic, stance), group in df_aio_src.groupby(
    ["topic", "subtopic", "stance"]
):
    domain_sets = [d for d in group["domain_set"] if d]
    if len(domain_sets) < 2:
        continue
    pair_jaccards = [
        j
        for j in (_jaccard(a, b) for a, b in combinations(domain_sets, 2))
        if j is not None
    ]
    if not pair_jaccards:
        continue
    baseline_rows.append(
        {
            "topic": topic,
            "subtopic": subtopic,
            "stance": stance,
            "jaccard": np.mean(pair_jaccards),
        }
    )

baseline_df = pd.DataFrame(baseline_rows)

if baseline_df.empty:
    print("No baseline data to aggregate yet.")
else:
    subtopic_leaning = (
        df_aio_src.dropna(subset=["subtopic", "pro_leaning"])
        .drop_duplicates("subtopic")
        .set_index("subtopic")["pro_leaning"]
    )
    baseline_df["pro_leaning"] = baseline_df["subtopic"].map(subtopic_leaning)
    baseline_df = baseline_df.dropna(subset=["pro_leaning"])

    n_stances = len(STANCE_ORDER)
    bar_w = 0.8 / n_stances
    offsets = np.linspace(-(n_stances - 1) / 2, (n_stances - 1) / 2, n_stances) * bar_w
    x = np.arange(len(LEANING_ORDER))

    topics_order = sorted(baseline_df["topic"].unique())
    n_topics = len(topics_order)
    ncols = 3
    nrows = -(-n_topics // ncols)

    fig, axes = plt.subplots(
        nrows, ncols, figsize=(ncols * 5, nrows * 4.5), sharey=True
    )
    axes = np.atleast_1d(axes).flatten()

    for ax, topic in zip(axes, topics_order):
        tgroup = baseline_df[baseline_df["topic"] == topic]
        pivot = (
            tgroup.groupby(["pro_leaning", "stance"])["jaccard"]
            .mean()
            .reset_index()
            .pivot(index="pro_leaning", columns="stance", values="jaccard")
            .reindex(LEANING_ORDER)
        )
        for stance, offset in zip(STANCE_ORDER, offsets):
            vals = (
                pivot[stance].values
                if stance in pivot.columns
                else np.zeros(len(LEANING_ORDER))
            )
            ax.bar(
                x + offset,
                vals,
                bar_w,
                color=PALETTE[stance],
                label=STANCE_LABELS[stance],
                edgecolor="white",
                linewidth=0.3,
            )
        ax.set_xticks(x)
        ax.set_xticklabels([lean.capitalize() for lean in LEANING_ORDER], fontsize=13)
        ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
        ax.set_ylim(0, 0.5)
        ax.tick_params(axis="y", labelsize=12)
        ax.set_title(topic, fontsize=15)

    for ax in axes[n_topics:]:
        ax.set_visible(False)

    for ax in axes[:n_topics:ncols]:
        ax.set_ylabel("Avg. Jaccard overlap of AIO-cited domains", fontsize=13)

    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(
        handles,
        labels,
        title="Stance",
        loc="upper center",
        bbox_to_anchor=(0.5, 1.06),
        ncol=3,
        fontsize=13,
        title_fontsize=14,
    )
    fig.suptitle(
        "Baseline: AIO source similarity within the same subtopic × stance\n"
        "(across query phrasings, subtopics merged by leaning)",
        y=1.14,
        fontsize=17,
    )
    plt.tight_layout()

    FIGURES_DIR = Path("figures")
    FIGURES_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(
        FIGURES_DIR / "aio_source_similarity_baseline_per_topic.pdf",
        bbox_inches="tight",
    )
    plt.show()

    print("\nAverage baseline overlap per topic × stance (Left+Right pooled):")
    print(baseline_df.groupby(["topic", "stance"])["jaccard"].mean().round(3).unstack())

In [ ]:
# ── 5k. Per topic: average source overlap, subtopics merged by leaning ────────
import numpy as np
from itertools import combinations
from pathlib import Path

if overlap_df.empty:
    print("No overlap data to aggregate yet.")
else:
    STANCE_ORDER = ["Pro", "Neutral", "Con"]
    LEANING_ORDER = ["Left", "Right"]
    pair_labels = [f"{s1} vs {s2}" for s1, s2 in combinations(STANCE_ORDER, 2)]
    PAIR_COLORS = {
        "Pro vs Neutral": "#8e44ad",
        "Pro vs Con": "#f39c12",
        "Neutral vs Con": "#16a085",
    }
    PAIR_DISPLAY = {
        "Pro vs Neutral": "Pros vs Neutral",
        "Pro vs Con": "Pros vs Cons",
        "Neutral vs Con": "Neutral vs Cons",
    }

    subtopic_leaning = (
        df_aio_src.dropna(subset=["subtopic", "pro_leaning"])
        .drop_duplicates("subtopic")
        .set_index("subtopic")["pro_leaning"]
    )

    overlap_df_lean = overlap_df.copy()
    overlap_df_lean["pro_leaning"] = overlap_df_lean["subtopic"].map(subtopic_leaning)
    overlap_df_lean = overlap_df_lean.dropna(subset=["pro_leaning"])

    n_pairs = len(pair_labels)
    bar_w = 0.8 / n_pairs
    offsets = np.linspace(-(n_pairs - 1) / 2, (n_pairs - 1) / 2, n_pairs) * bar_w
    x = np.arange(len(LEANING_ORDER))

    topics_order = sorted(overlap_df_lean["topic"].unique())
    n_topics = len(topics_order)
    ncols = 3
    nrows = -(-n_topics // ncols)

    fig, axes = plt.subplots(
        nrows, ncols, figsize=(ncols * 5, nrows * 4.5), sharey=True
    )
    axes = np.atleast_1d(axes).flatten()

    for ax, topic in zip(axes, topics_order):
        tgroup = overlap_df_lean[overlap_df_lean["topic"] == topic]
        pivot = (
            tgroup.groupby(["pro_leaning", "pair"])["jaccard"]
            .mean()
            .reset_index()
            .pivot(index="pro_leaning", columns="pair", values="jaccard")
            .reindex(LEANING_ORDER)
        )
        for pair, offset in zip(pair_labels, offsets):
            vals = (
                pivot[pair].values
                if pair in pivot.columns
                else np.zeros(len(LEANING_ORDER))
            )
            ax.bar(
                x + offset,
                vals,
                bar_w,
                color=PAIR_COLORS[pair],
                label=PAIR_DISPLAY[pair],
                edgecolor="white",
                linewidth=0.3,
            )
        ax.set_xticks(x)
        ax.set_xticklabels([lean.capitalize() for lean in LEANING_ORDER], fontsize=13)
        ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
        ax.set_ylim(0, 0.5)
        ax.tick_params(axis="y", labelsize=12)
        ax.set_title(topic, fontsize=15)

    for ax in axes[n_topics:]:
        ax.set_visible(False)

    for ax in axes[:n_topics:ncols]:
        ax.set_ylabel("Avg. Jaccard overlap of AIO-cited domains", fontsize=13)

    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(
        handles,
        labels,
        title="Formulation pair",
        loc="upper center",
        bbox_to_anchor=(0.5, 1.06),
        ncol=3,
        fontsize=13,
        title_fontsize=14,
    )
    fig.suptitle(
        "AIO source overlap between formulations — subtopics merged by leaning (per topic)",
        y=1.1,
        fontsize=17,
    )
    plt.tight_layout()

    FIGURES_DIR = Path("figures")
    FIGURES_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(
        FIGURES_DIR / "aio_source_overlap_leaning_per_topic.pdf",
        bbox_inches="tight",
    )
    plt.show()

In [ ]:
# ── 5l. Overall: average source overlap, subtopics merged by leaning (all topics) ──
import numpy as np
from itertools import combinations

if overlap_df.empty:
    print("No overlap data to aggregate yet.")
else:
    STANCE_ORDER = ["Pro", "Neutral", "Con"]
    LEANING_ORDER = ["Left", "Right"]
    pair_labels = [f"{s1} vs {s2}" for s1, s2 in combinations(STANCE_ORDER, 2)]
    PAIR_COLORS = {
        "Pro vs Neutral": "#8e44ad",
        "Pro vs Con": "#f39c12",
        "Neutral vs Con": "#16a085",
    }
    PAIR_DISPLAY = {
        "Pro vs Neutral": "Pros vs Neutral",
        "Pro vs Con": "Pros vs Cons",
        "Neutral vs Con": "Neutral vs Cons",
    }

    subtopic_leaning = (
        df_aio_src.dropna(subset=["subtopic", "pro_leaning"])
        .drop_duplicates("subtopic")
        .set_index("subtopic")["pro_leaning"]
    )

    overlap_df_lean_all = overlap_df.copy()
    overlap_df_lean_all["pro_leaning"] = overlap_df_lean_all["subtopic"].map(
        subtopic_leaning
    )
    overlap_df_lean_all = overlap_df_lean_all.dropna(subset=["pro_leaning"])

    simple_pivot = (
        overlap_df_lean_all.groupby(["pro_leaning", "pair"])["jaccard"]
        .mean()
        .reset_index()
        .pivot(index="pro_leaning", columns="pair", values="jaccard")
        .reindex(LEANING_ORDER)
    )

    n_pairs = len(pair_labels)
    bar_w = 0.8 / n_pairs
    offsets = np.linspace(-(n_pairs - 1) / 2, (n_pairs - 1) / 2, n_pairs) * bar_w
    x = np.arange(len(LEANING_ORDER))

    fig, ax = plt.subplots(figsize=(6, 4.5))
    for pair, offset in zip(pair_labels, offsets):
        vals = (
            simple_pivot[pair].values
            if pair in simple_pivot.columns
            else np.zeros(len(LEANING_ORDER))
        )
        ax.bar(
            x + offset,
            vals,
            bar_w,
            color=PAIR_COLORS[pair],
            label=PAIR_DISPLAY[pair],
            edgecolor="white",
            linewidth=0.3,
        )
    ax.set_xticks(x)
    ax.set_xticklabels([lean.capitalize() for lean in LEANING_ORDER])
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
    ax.set_ylim(0, 0.5)
    ax.set_ylabel("Avg. Jaccard overlap of AIO-cited domains")
    ax.set_title(
        "AIO source overlap between formulations — subtopics merged by leaning (all topics)"
    )
    ax.legend(title="Formulation pair")
    plt.tight_layout()
    plt.show()

    print("\nAverage overlap per leaning × pair:")
    print(simple_pivot.round(3).to_string())

In [ ]:
# ── 5m. Matched query-pair comparison (symmetric groups) ──────────────────────
# `symmetric_groups_detail.csv` pairs near-identical queries that differ mainly
# by their pro/contro (and sometimes neutrale) framing — e.g. "...è sicuro" vs
# "...è pericoloso". Comparing AIO-cited domains within these matched pairs is a
# tighter test of framing sensitivity than 5h-5j above: those sections pool
# every query sharing a subtopic+stance, while here each comparison is between
# two (or three) queries that are near word-for-word identical apart from the
# framing itself.
sym_groups = pd.read_csv(ROOT / "queries" / "symmetric_groups_detail.csv", sep=";")

df_real_aio = df[df["has_ai_overview"]].copy()
df_real_aio["domain_set"] = df_real_aio["aio_domains"].apply(_domain_set)
df_real_aio["has_real_content"] = df_real_aio.apply(
    lambda r: _has_real_aio_content(r["domain_set"], r.get("aio_text")), axis=1
)
# rows where the AIO "shell" appeared but the second-stage content fetch
# returned nothing (no text, no sources) don't count as a usable AIO for
# this comparison — see the note in 5h above.
df_real_aio = df_real_aio[df_real_aio["has_real_content"]]

query_domains = df_real_aio.groupby("query")["domain_set"].apply(
    lambda sets: set().union(*sets)
)
query_has_real_aio = df_real_aio.groupby("query").size() > 0
subtopic_topic = (
    df.dropna(subset=["subtopic", "topic"])
    .drop_duplicates("subtopic")
    .set_index("subtopic")["topic"]
)


def _lookup(query):
    """(domain_set, had_real_aio) for a query.

    had_real_aio is False if the query was never collected, the AIO never
    appeared, or the AIO fetch returned an empty stub (no text, no sources —
    a failed second-stage content fetch, not a genuine zero-source AIO).
    """
    if query not in query_has_real_aio.index or not query_has_real_aio[query]:
        return set(), False
    return query_domains.get(query, set()), True


rows = []
for _, g in sym_groups.iterrows():
    pro_domains, pro_ok = _lookup(g["pro_query"])
    contro_domains, contro_ok = _lookup(g["contro_query"])
    both_ok = pro_ok and contro_ok

    row = {
        "group_id": g["group_id"],
        "topic": subtopic_topic.get(g["subtopic"]),
        "subtopic": g["subtopic"],
        "pro_query": g["pro_query"],
        "contro_query": g["contro_query"],
        "pro_has_aio": pro_ok,
        "contro_has_aio": contro_ok,
        "jaccard_pro_contro": _jaccard(pro_domains, contro_domains)
        if both_ok
        else None,
        "only_in_pro": sorted(pro_domains - contro_domains) if both_ok else None,
        "only_in_contro": sorted(contro_domains - pro_domains) if both_ok else None,
    }

    neutrale_query = g.get("neutrale_query")
    if isinstance(neutrale_query, str) and neutrale_query:
        neutrale_domains, neutrale_ok = _lookup(neutrale_query)
        row["neutrale_query"] = neutrale_query
        row["neutrale_has_aio"] = neutrale_ok
        row["jaccard_pro_neutrale"] = (
            _jaccard(pro_domains, neutrale_domains)
            if (pro_ok and neutrale_ok)
            else None
        )
        row["jaccard_contro_neutrale"] = (
            _jaccard(contro_domains, neutrale_domains)
            if (contro_ok and neutrale_ok)
            else None
        )

    rows.append(row)

sym_comparison = pd.DataFrame(rows)
n_comparable = sym_comparison["jaccard_pro_contro"].notna().sum()
print(f"Symmetric groups loaded          : {len(sym_comparison)}")
print(f"Comparable (both sides had real AIO content) : {n_comparable}")
if n_comparable:
    print(
        f"Mean pro-vs-contro Jaccard       : {sym_comparison['jaccard_pro_contro'].mean():.3f}"
    )
    print(
        f"Median pro-vs-contro Jaccard     : {sym_comparison['jaccard_pro_contro'].median():.3f}"
    )

display(
    sym_comparison[
        [
            "group_id",
            "topic",
            "subtopic",
            "pro_query",
            "contro_query",
            "jaccard_pro_contro",
        ]
    ].sort_values("jaccard_pro_contro")
)

In [ ]:
# ── 5n. Most framing-sensitive matched pairs + distribution ──────────────────
from pathlib import Path

comparable = sym_comparison.dropna(subset=["jaccard_pro_contro"])

if comparable.empty:
    print("No matched pairs with AIO on both sides yet.")
else:
    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.hist(
        comparable["jaccard_pro_contro"], bins=15, color="#8e44ad", edgecolor="white"
    )

    ax.set_xlabel("Jaccard overlap of AIO-cited domains (pro vs contro)")
    ax.set_ylabel("Number of matched query pairs")
    ax.set_title("AIO source overlap (near-identical wording, framing swapped)")
    plt.tight_layout()

    FIGURES_DIR = Path("figures")
    FIGURES_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(FIGURES_DIR / "sym_groups_pro_contro_jaccard.pdf", bbox_inches="tight")
    plt.show()

    N_LOWEST = 15
    print(
        f"\n{N_LOWEST} most framing-sensitive matched pairs (lowest pro-vs-contro overlap):"
    )
    for _, r in comparable.sort_values("jaccard_pro_contro").head(N_LOWEST).iterrows():
        only_pro = r["only_in_pro"]
        only_contro = r["only_in_contro"]
        print(
            f"\n[{r['group_id']}] {r['topic']} / {r['subtopic']}  (Jaccard = {r['jaccard_pro_contro']:.2f})"
        )
        print(f"   pro    : {r['pro_query']}")
        print(f"   contro : {r['contro_query']}")
        print(
            f"   only in pro    ({len(only_pro)}): {only_pro[:8]}{' …' if len(only_pro) > 8 else ''}"
        )
        print(
            f"   only in contro ({len(only_contro)}): {only_contro[:8]}{' …' if len(only_contro) > 8 else ''}"
        )

In [ ]:
# ── 5o. Neutral formulation vs Pro/Con — matched groups with a neutral query ──
# Of the 82 symmetric groups, 15 have a matched neutral-formulation query
# (e.g. "equidistanza tra fascismo e antifascismo significato"). This shows,
# for each of those 15, how much the neutral AIO's cited domains overlap with
# the pro AIO's domains vs. the contro AIO's domains.
from pathlib import Path
import numpy as np

neutral_groups = (
    sym_comparison[sym_comparison["neutrale_query"].notna()].copy()
    if "neutrale_query" in sym_comparison.columns
    else pd.DataFrame()
)

if neutral_groups.empty:
    print("No matched groups with a neutral query yet.")
else:
    neutral_groups["label"] = (
        neutral_groups["group_id"] + " · " + neutral_groups["subtopic"]
    )
    neutral_groups = neutral_groups.sort_values(["subtopic", "group_id"])

    n_missing = (
        neutral_groups[["jaccard_pro_neutrale", "jaccard_contro_neutrale"]]
        .isna()
        .any(axis=1)
        .sum()
    )
    if n_missing:
        print(
            f"Note: {n_missing} of {len(neutral_groups)} groups are missing a pro-vs-neutral or "
            "contro-vs-neutral value (one side had no usable AIO content) — left as a gap, not a 0."
        )

    labels = neutral_groups["label"].tolist()
    y = np.arange(len(labels))
    bar_h = 0.35

    fig, ax = plt.subplots(figsize=(9, max(3, len(labels) * 0.45)))
    ax.barh(
        y + bar_h / 2,
        neutral_groups["jaccard_pro_neutrale"],
        bar_h,
        color=PALETTE["Pro"],
        label="Neutral vs Pros",
        edgecolor="white",
        linewidth=0.3,
    )
    ax.barh(
        y - bar_h / 2,
        neutral_groups["jaccard_contro_neutrale"],
        bar_h,
        color=PALETTE["Con"],
        label="Neutral vs Cons",
        edgecolor="white",
        linewidth=0.3,
    )
    ax.set_yticks(y)
    ax.set_yticklabels(labels, fontsize=8)
    ax.invert_yaxis()
    ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
    ax.set_xlim(0, 1.05)
    ax.set_xlabel("Jaccard overlap of AIO-cited domains vs. neutral query")
    ax.set_title(f"Neutral-query source overlap for {len(labels)} matched groups")
    ax.legend(fontsize=9)
    plt.tight_layout()

    FIGURES_DIR = Path("figures")
    FIGURES_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(
        FIGURES_DIR / "sym_groups_neutral_vs_pro_con_jaccard.pdf", bbox_inches="tight"
    )
    plt.show()